# Gradient descent
**Machine Learning · Week 2 (Regression and gradient descent) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/ml/w2d2.html

When there is no formula for the best knobs, walk downhill: compute the slope (gradient) of the
loss and take a small step the other way. Repeat.

$$w \leftarrow w - \eta\,\frac{dL}{dw}$$

In [ ]:
def loss(w):
    return (w - 3) ** 2 + 1          # a simple bowl with its bottom at w = 3

def grad(w):
    return 2 * (w - 3)               # its derivative (slope)

w, lr = -2.0, 0.1                    # start far away; lr is the learning rate (eta)
for step in range(8):
    print(f"step {step}: w = {w:6.3f}   loss = {loss(w):7.3f}   slope = {grad(w):7.3f}")
    w = w - lr * grad(w)

Each step moves `w` closer to 3, and the steps get smaller as the slope flattens.
Now let's compare three learning rates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ws = np.linspace(-3, 9, 200)
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True, layout="constrained")
for ax, lr in zip(axes, [0.05, 0.4, 1.02]):
    w = -2.0
    path = [w]
    for _ in range(12):
        w = w - lr * grad(w)
        path.append(w)
    path = np.array(path)
    ax.plot(ws, loss(ws), color="#b9a7ff", lw=3)
    ax.plot(path, loss(path), "o-", color="#ff8a50", ms=5, lw=1.5)
    ax.set_title(f"learning rate = {lr}")
    ax.set_xlim(-3, 9)
    ax.set_ylim(0, 45)
    ax.set_xlabel("w")
axes[0].set_ylabel("loss")
plt.show()

Too small: slow. Just right: quick. Too big: it overshoots back and forth and flies away.

## Gradient descent for the pizza line
The MSE of a line has two knobs, so the gradient has two parts:
$\frac{\partial L}{\partial w} = \frac{2}{n}\sum(\hat y_i - y_i)\,x_i$ and
$\frac{\partial L}{\partial b} = \frac{2}{n}\sum(\hat y_i - y_i)$.
We first **scale** the feature (mean 0, spread 1): gradient descent works much better
when inputs are not huge numbers like 30 cm.

In [ ]:
rng = np.random.default_rng(3)
x = rng.uniform(20, 40, size=25)
y = 0.55 * x - 2 + rng.normal(0, 1.4, size=25)
xs = (x - x.mean()) / x.std()               # scaled feature

w, b, lr = 0.0, 0.0, 0.1
history = []
for step in range(60):
    error = (w * xs + b) - y
    dw = 2 * np.mean(error * xs)
    db = 2 * np.mean(error)
    w, b = w - lr * dw, b - lr * db
    history.append(np.mean(((w * xs + b) - y) ** 2))

print(f"after 60 steps: w = {w:.3f}, b = {b:.3f}, MSE = {history[-1]:.3f}")
exact = np.polyfit(xs, y, 1)
print(f"exact answer  : w = {exact[0]:.3f}, b = {exact[1]:.3f}")

In [ ]:
plt.plot(history, color="#7b5cfa")
plt.yscale("log")
plt.xlabel("step")
plt.ylabel("MSE (log scale)")
plt.title("A training curve")
plt.show()

## Your turn: why we scaled the feature
Another pizza place charges an \$8 base fee plus 0.5 \$/cm. Complete `train` with the two gradients and
the update from the cell above, then compare the scaled feature `xs` with the raw diameters `x`.

In [ ]:
y_fee = 0.5 * x + 8 + rng.normal(0, 1.4, size=25)       # $8 base fee

def train(feature, target, lr, steps):
    w, b = 0.0, 0.0
    for _ in range(steps):
        error = (w * feature + b) - target
        # TODO: compute dw and db, then update w and b (copy from the training loop above)
    return np.mean(((w * feature + b) - target) ** 2)

for lr in [0.1, 0.01, 0.001]:
    print(f"lr = {lr:<6} MSE after 60 steps: scaled {train(xs, y_fee, lr, 60):8.2f}   raw {train(x, y_fee, lr, 60):8.2f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def train(feature, target, lr, steps):
    w, b = 0.0, 0.0
    with np.errstate(over="ignore", invalid="ignore"):      # diverging runs overflow; we report them instead
        for _ in range(steps):
            error = (w * feature + b) - target
            dw = 2 * np.mean(error * feature)
            db = 2 * np.mean(error)
            w, b = w - lr * dw, b - lr * db
        return np.mean(((w * feature + b) - target) ** 2)

def show(v):
    return "diverged" if not np.isfinite(v) or v > 1e6 else f"{v:8.2f}"

for lr in [0.1, 0.01, 0.001]:
    print(f"lr = {lr:<6} MSE after 60 steps: scaled {show(train(xs, y_fee, lr, 60))}   raw {show(train(x, y_fee, lr, 60))}")
for steps in [600, 6000, 60000]:
    print(f"raw diameters, lr = 0.001, {steps:5d} steps: MSE = {show(train(x, y_fee, 0.001, steps))}")
print("\nThe best possible MSE is about", round(np.mean((np.polyval(np.polyfit(x, y_fee, 1), x) - y_fee) ** 2), 2))
print("Raw diameters (around 30) turn the loss bowl into a long, steep, narrow valley. A learning rate small")
print("enough not to fly off across the valley creeps along it, so the intercept needs tens of thousands of steps.")
print("Scaling makes the bowl round, and 60 steps are plenty.")